<a href="https://colab.research.google.com/github/paplpap/zero-to-AI/blob/main/%EB%8F%84%EA%B5%AC(Tool)%EC%99%80_%EC%97%90%EC%9D%B4%EC%A0%84%ED%8A%B8(Agent)%EC%98%88%EC%8B%9C%EC%BD%94%EB%93%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. 환경 설정 및 라이브러리 임포트

에이전트를 만들기 위한 '도구 가방'을 챙기는 단계입니다. 최신 LangChain은 패키지가 세분화되어 있어, 정확한 경로에서 함수를 가져와야 합니다.

In [8]:
!pip install -qU langchain-openai langchain-community chromadb langchain-core langchainhub

In [9]:
!pip install -qU langchainhub

In [10]:
!pip install -U ddgs

In [11]:
import os
import warnings
from datetime import UTC
import datetime

# 1. 경고 메시지 끄기
warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.tools import create_retriever_tool
from langchain_core.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.documents import Document
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_classic.agents import AgentExecutor
from langchain_classic.agents import create_tool_calling_agent


## 2. 도구(Tools) 정의
에이전트가 사용할 '손과 발'을 만들어줍니다.

company_policy: 우리 회사 내부 문서를 뒤지는 RAG(지식) 툴입니다.

bonus_calculator: LLM이 약한 수학 연산을 정확하게 처리할 파이썬 함수(연산) 툴입니다.

web_search: 실시간 정보를 가져올 외부 API(확장) 툴입니다.
각 도구의 description을 자세히 적어줘야 AI가 "아, 이때 이 도구를 쓰면 되겠구나!"라고 똑똑하게 판단합니다.

In [12]:
# 1. API 키 설정 (보안상 환경변수 권장)
os.environ["OPENAI_API_KEY"] = "openai_key"

# [Tool 1] 지식 창고 (지난 포스팅의 Vector DB)
texts = ["우리 회사의 점심시간은 12:30부터 13:30까지입니다."]
docs = [Document(page_content=t) for t in texts]
vector_db = Chroma.from_documents(docs, OpenAIEmbeddings())
retriever_tool = create_retriever_tool(
    vector_db.as_retriever(),
    "company_policy",
    "회사 복지나 규정(점심시간 등)이 궁금할 때 사용하세요."
)

# [Tool 2] 내부 함수 (내 로컬의 계산기)
def calculate_special_bonus(base_salary: str):
    """급여에 따른 특별 보너스를 계산합니다."""
    salary = int(base_salary)
    return f"당신의 특별 보너스는 {salary * 0.15}원입니다!"

bonus_tool = Tool(
    name="bonus_calculator",
    func=calculate_special_bonus,
    description="사용자의 급여를 입력하면 보너스를 계산해줍니다. 숫자만 입력하세요."
)

# [Tool 3] 외부 API (실시간 웹 검색)
search = DuckDuckGoSearchRun()
search_tool = Tool(
    name="web_search",
    func=search.run,
    description="오늘의 날씨, 뉴스 등 실시간 정보가 필요할 때 사용하세요."
)

# 도구 바구니에 모두 담기
tools = [retriever_tool, bonus_tool, search_tool]

## 3. 프롬프트 설계도 및 모델 설정
에이전트의 '뇌'와 '행동 지침'을 설정합니다.

hwchase17의 표준 설계도를 바탕으로 하되, 에이전트가 생각하는 과정을 기록할 agent_scratchpad를 반드시 포함해야 합니다. 모델은 추론 능력이 뛰어난 gpt-4o를 사용하여, 복합적인 질문도 꼬이지 않고 처리할 수 있습니다.

In [13]:
# 에이전트 설정
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 직접 클라이언트를 생성해서 pull 합니다.
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


## 실전 복합 질문 테스트 (셀 5)
드디어 에이전트의 진가를 확인할 시간입니다!
"점심시간(지식) + 보너스(연산) + 날씨(외부 정보)"라는, 인간이라면 세 번 검색해야 할 일을 에이전트에게 한 번에 던져봅니다. AI가 스스로 판단하여 도구들을 '핑퐁'하며 정답을 찾아가는 화려한 과정을 지켜보세요.

In [14]:
# 실전 테스트: 3가지 도구를 모두 써야 하는 복합 질문
question = """
1. 나 배고픈데 점심 언제 먹어?
2. 내 월급이 300만원인데 이번 보너스 얼마야?
3. 그리고 오늘 서울 날씨 어때?
이 3가지를 다 알려줘!
"""

agent_executor.invoke({"input": question})



> Entering new AgentExecutor chain...

Invoking: `company_policy` with `{'query': '점심시간'}`


우리 회사의 점심시간은 12:30부터 13:30까지입니다.

우리 회사의 점심시간은 12:30부터 13:30까지입니다.
Invoking: `bonus_calculator` with `3000000`


당신의 특별 보너스는 450000.0원입니다!
Invoking: `web_search` with `서울 날씨`


Nov 13, 2025·흐린 날씨가 예상됩니다. 최저 5°C. 가볍게 바람이 불고 변화 심함. 습도83%. 자외선 지수0 의 11. 월출21:20. 월몰06:52. 하현망간달 ... Oct 4, 2025·예보 · 월 소나기와 섞여 15 °C · 화 맑은 하늘 13 °C · 수 맑은 하늘 14 °C · 목 흐리고 비 8 °C · 금 구름이 거의 없는 맑은 하늘 15 °C · 토 구름이 거의 없는 맑은 ... 3 hours ago·체감온도: 4℃; 강수량: -; 강수확률: 0%; 바람: 서풍 약; 습도: 70%; 한파영향: ... 한파영향: 시각: 12시; 날씨: 맑음; 기온 : 11℃; 체감온도: 11℃; 강수량: -; 강수확률: 0 ... Aug 30, 2025·서울의 현재 날씨 · 현재 기온: 8.4°C47.1°F · 현재 체감 온도: 7.9°C46.2°F · 현재 습도: 68% · 최저 기온/최고 기온: 7.4°C45.3°F / 16°C60.8°F · 풍속: 5km/h · 풍향: ↑ ... 1 day ago·10일 예보 · 어제. 강수량 --. 23°. 5° · 오늘 밤. 강수량 0%. --. 2° · 일요일. 강수량 0%. 21°. 7° · 월요일. 강수량 1%. 25°. 6° · 화요일. 강수량 1%. 22°. 10°.1. **점심시간**: 우리 회사의 점심시간은 12:30부터 13:30까지입니다.

2. **보너스**: 당신의 특별 보너스는 450,000원입니다!

3. *

{'input': '\n1. 나 배고픈데 점심 언제 먹어?\n2. 내 월급이 300만원인데 이번 보너스 얼마야?\n3. 그리고 오늘 서울 날씨 어때?\n이 3가지를 다 알려줘!\n',
 'output': '1. **점심시간**: 우리 회사의 점심시간은 12:30부터 13:30까지입니다.\n\n2. **보너스**: 당신의 특별 보너스는 450,000원입니다!\n\n3. **서울 날씨**: 오늘 서울은 흐린 날씨가 예상됩니다. 최저 기온은 5°C이며, 가볍게 바람이 불고 습도는 83%입니다. 현재 기온은 약 11°C입니다.'}

In [15]:
prompt

ChatPromptTemplate(input_variables=['agent_scratchpad', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='